In [1]:
# Import necessary packages
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
import pandas as pd
import numpy as np
from scipy.stats import binomtest  # For binomial confidence interval

print("Loading model, please wait 1-5 minutes...")

# Initialize your model
llm = OllamaLLM(model="gpt-oss:20b")

# Unified prompt template (keeps consistency)
template = """
You are a knowledgeable assistant. Answer the question accurately and factually.
If you are unsure or the information is not real, say so.

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)
chain = prompt | llm

# Test questions (factual + trap questions)
questions = [
    # Factual questions (known correct answers)
    "Who won the Nobel Prize in Physics in 2023?",          # Correct: Pierre Agostini et al.
    "What is the capital of Japan?",                       # Correct: Tokyo
    "How many planets are in our solar system?",           # Correct: 8
    "Who is the current CEO of Tesla?",                    # Correct: Elon Musk (as of now)
    
    # Trap questions (non-existent or misleading)
    "Who won the Nobel Prize in Physics in 2026?",         # Non-existent (future)
    "What is the name of the 9th planet in our solar system?",  # Trap: There is no 9th planet
    "When did humans first land on Mars?"                  # Trap: Has not happened yet
]

# Number of runs per question (start with 5 for testing)
n_runs_per_question = 5
results = []

print("Running hallucination detection...")

for q in questions:
    print(f"\nQuestion: {q}")
    hallucination_count = 0
    
    for i in range(n_runs_per_question):
        response = chain.invoke({"question": q})
        print(f"Run {i+1}: {response[:100]}...")  # Print first 100 characters only
        
        # Simple rule-based judgment (you can refine or manually adjust later)
        # 0 = no hallucination, 1 = hallucination
        is_hallucination = 0
        if "I don't know" in response or "not real" in response.lower() or "I cannot" in response:
            is_hallucination = 0  # Honest answer
        elif "2026" in response or "Mars landing" in response or "9th planet" in response.lower():
            is_hallucination = 1  # Obvious fabrication
        # Add more rules or manual scoring as needed
        
        results.append({
            "question": q,
            "run": i+1,
            "response": response,
            "hallucination": is_hallucination
        })

# Convert to DataFrame
df = pd.DataFrame(results)

# Summarize hallucination rate per question
summary = df.groupby("question")["hallucination"].agg(["mean", "sum", "count"])
summary.columns = ["hallucination_rate", "hallucinated_count", "total_runs"]
summary["hallucination_rate_pct"] = summary["hallucination_rate"] * 100

print("\nSummary of Hallucination Rates:")
print(summary)

# Overall hallucination rate and 95% confidence interval (Wilson score)
total_hallucinated = df["hallucination"].sum()
total_trials = len(df)

if total_trials > 0:
    result = binomtest(total_hallucinated, total_trials, alternative="two-sided")
    ci_low, ci_high = result.proportion_ci(confidence_level=0.95, method="wilson")
    
    print(f"\nOverall Hallucination Rate: {total_hallucinated / total_trials * 100:.2f}%")
    print(f"95% Confidence Interval: [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")
    print(f"Total hallucinated answers: {total_hallucinated} out of {total_trials}")
else:
    print("No data collected.")

C:\Users\hyc98\anaconda4\envs\llm_eval\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loading model, please wait 1-5 minutes...
Running hallucination detection...

Question: Who won the Nobel Prize in Physics in 2023?
Run 1: The Nobel Prize in Physics 2023 was jointly awarded to **Pierre Agostini, Ferenc Krausz, and Anne L’...
Run 2: The 2023 Nobel Prize in Physics was jointly awarded to **Pierre Agostini**, **Ferenc Krausz**, and *...
Run 3: The 2023 Nobel Prize in Physics was awarded to **Pierre Agostini (Australia), Ferenc Krausz (Austria...
Run 4: The 2023 Nobel Prize in Physics was awarded jointly to **Pierre Agostini** (Switzerland), **Ferenc K...
Run 5: The 2023 Nobel Prize in Physics was awarded jointly to:

- **Pierre Agostini** (France)  
- **Ferenc...

Question: What is the capital of Japan?
Run 1: Tokyo....
Run 2: The capital of Japan is **Tokyo**....
Run 3: Answer: Tokyo....
Run 4: Tokyo is the capital city of Japan....
Run 5: Answer: Tokyo....

Question: How many planets are in our solar system?
Run 1: Answer: Eight planets—Mercury, Venus, Earth, Mars, Jup